<a href="https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/barbie_network_analysis_2026.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Character networks: Barbie
Compare **co-presence** (shared scenes, undirected) with **dialogue** (speaker → addressee, directed).

The embedded edge lists are inherited teaching examples from the 2025 course notebook. Their counts have not been verified against the screenplay: use them to learn the method, not as established findings about the film. Check passages before making literary claims.

Run from the top. For your own project, use two CSV edge lists. This notebook does **not** extract reliable relationships automatically from a PDF.


---
# Setup and Data Loading

In [ ]:
# Colab includes these libraries; no package installation is needed there.
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import io
from IPython.display import display

plt.rcParams.update({'figure.dpi': 110})
print('NetworkX', nx.__version__)


## Data
Each CSV needs `Source,Target,Weight`. Use consistent character names and positive numeric weights.
- Co-presence: count shared scenes, not minutes of screen time.
- Dialogue: count addressed speech turns; direction matters. Consecutive speakers are not automatically speaking to each other.

Run the next cell for the teaching data. Set `USE_UPLOADS = True` to upload both CSVs instead. Repeated pairs are summed; check they represent additional observations rather than duplicate counts.


In [ ]:
# ==============================================================================
# CO-PRESENCE DATA (undirected: A shares scene with B)
# ==============================================================================
copresence_data = """Source,Target,Weight
Barbie Margot,Ken Ryan Gosling,20
Barbie Margot,Gloria,12
Barbie Margot,Sasha,11
Barbie Margot,Weird Barbie,9
Barbie Margot,Ken Simu Liu,8
Barbie Margot,Mattel CEO,6
Barbie Margot,President Barbie,5
Barbie Margot,Allan,4
Barbie Margot,Ruth,3
Ken Ryan Gosling,Ken Simu Liu,10
Ken Ryan Gosling,President Barbie,7
Ken Ryan Gosling,Doctor Barbie,6
Ken Ryan Gosling,Physicist Barbie,6
Ken Ryan Gosling,Ken Kingsley,5
Ken Ryan Gosling,Gloria,4
Ken Ryan Gosling,Sasha,4
Ken Ryan Gosling,Mattel CEO,4
Ken Ryan Gosling,Weird Barbie,2
Gloria,Sasha,8
Gloria,Mattel CEO,5
Gloria,Weird Barbie,3
Gloria,Nerdy Well Meaning Dad,3
Sasha,Nerdy Well Meaning Dad,3
Mattel CEO,Aaron (Mattel Employee),4
Mattel CEO,Other Board Members,3
President Barbie,Doctor Barbie,5
President Barbie,Physicist Barbie,5
President Barbie,Judge Barbie,5
Doctor Barbie,Physicist Barbie,4
Doctor Barbie,Judge Barbie,4
Physicist Barbie,Judge Barbie,4
Ken Simu Liu,Ken Kingsley,4
Ken Simu Liu,Ken Ncuti,4
Ken Simu Liu,Ken Scott,4
Allan,Skipper,3
Allan,Midge,2
Allan,Weird Barbie,4
Skipper,Midge,2
Midge,Allan,1
Judge Barbie,Journalist Barbie,2
"""

# ==============================================================================
# DIALOGUE DATA (directed: Source speaks TO Target)
# ==============================================================================
dialogue_data = """Source,Target,Weight
Barbie Margot,Ken Ryan Gosling,35
Barbie Margot,Gloria,25
Barbie Margot,Sasha,15
Barbie Margot,Weird Barbie,12
Barbie Margot,Mattel CEO,8
Barbie Margot,Ruth,6
Barbie Margot,President Barbie,5
Barbie Margot,Allan,4
Ken Ryan Gosling,Barbie Margot,40
Ken Ryan Gosling,Ken Simu Liu,12
Ken Ryan Gosling,Mattel CEO,3
Ken Ryan Gosling,Ken Ncuti,3
Ken Ryan Gosling,Ken Kingsley,3
Gloria,Barbie Margot,30
Gloria,Sasha,10
Gloria,Nerdy Well Meaning Dad,5
Gloria,Mattel CEO,4
Sasha,Barbie Margot,10
Sasha,Gloria,5
Sasha,Nerdy Well Meaning Dad,3
Weird Barbie,Barbie Margot,15
Weird Barbie,Allan,5
Mattel CEO,Barbie Margot,10
Mattel CEO,Gloria,5
Mattel CEO,Aaron (Mattel Employee),4
President Barbie,Ken Ryan Gosling,5
President Barbie,Doctor Barbie,3
President Barbie,Judge Barbie,3
Ruth,Barbie Margot,10
Allan,Weird Barbie,4
Allan,Gloria,3
Ken Simu Liu,Ken Ryan Gosling,10
Ken Simu Liu,Ken Ncuti,4
Doctor Barbie,President Barbie,3
Physicist Barbie,President Barbie,2
Judge Barbie,President Barbie,2
Skipper,Allan,1
Midge,Skipper,1
"""

# Set True only when you have two CSV edge lists ready.
USE_UPLOADS = False
if USE_UPLOADS:
    from google.colab import files
    print('Upload co-presence CSV first (one file).')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one co-presence CSV.')
    copresence_data = next(iter(uploaded.values())).decode('utf-8-sig')
    print('Upload dialogue CSV next (one file).')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one dialogue CSV.')
    dialogue_data = next(iter(uploaded.values())).decode('utf-8-sig')

def load_edges(text, directed=False):
    frame = pd.read_csv(io.StringIO(text))
    if not {'Source', 'Target', 'Weight'}.issubset(frame.columns):
        raise ValueError('CSV needs Source,Target,Weight columns.')
    frame = frame[['Source', 'Target', 'Weight']].copy()
    if frame.empty or frame.isna().any().any():
        raise ValueError('Edge list is empty or contains missing values.')
    for col in ['Source', 'Target']:
        frame[col] = frame[col].astype(str).str.strip()
        if frame[col].eq('').any():
            raise ValueError('Character names cannot be blank.')
    frame['Weight'] = pd.to_numeric(frame['Weight'], errors='raise')
    if not np.isfinite(frame['Weight']).all() or (frame['Weight'] <= 0).any():
        raise ValueError('Weights must be finite positive numbers.')
    if (frame['Source'] == frame['Target']).any():
        raise ValueError('Remove self-links before running.')
    if not directed:
        pairs = [sorted([a, b]) for a, b in zip(frame.Source, frame.Target)]
        frame[['Source', 'Target']] = pairs
    if frame.duplicated(['Source', 'Target']).any():
        print('Repeated pairs combined by summing weights. Check for accidental double counting.')
    frame = frame.groupby(['Source', 'Target'], as_index=False)['Weight'].sum()
    graph = nx.from_pandas_edgelist(frame, 'Source', 'Target', edge_attr='Weight',
                                   create_using=nx.DiGraph() if directed else nx.Graph())
    return frame, graph

df_copresence, G = load_edges(copresence_data)
df_dialogue, G_dialogue = load_edges(dialogue_data, directed=True)
print(f'Co-presence: {len(G)} characters, {G.number_of_edges()} relationships')
print(f'Dialogue: {len(G_dialogue)} characters, {G_dialogue.number_of_edges()} directed relationships')
display(df_copresence.head())


# 1. Co-presence
**Degree** counts partners. **Strength** sums shared-scene counts across partners. Betweenness and closeness below use unweighted paths; eigenvector uses the scene-count weights. More shared scenes are stronger ties, not longer distances.


## 1.1 Co-Presence Centrality and Global Metrics

In [ ]:
# === Centrality Calculations ===
degree = dict(G.degree())
strength = dict(G.degree(weight='Weight'))
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector = nx.eigenvector_centrality(G, weight='Weight', max_iter=1000)

co_presence_df = pd.DataFrame({
    'Character': list(G.nodes()),
    'Degree': [degree[n] for n in G.nodes()],
    'Strength': [strength[n] for n in G.nodes()],
    'Betweenness': [betweenness[n] for n in G.nodes()],
    'Closeness': [closeness[n] for n in G.nodes()],
    'Eigenvector': [eigenvector[n] for n in G.nodes()]
}).sort_values('Strength', ascending=False)

# === Global Metrics ===
transitivity = nx.transitivity(G)
components = list(nx.connected_components(G))
largest = G.subgraph(max(components, key=len)).copy()
avg_path = nx.average_shortest_path_length(largest) if len(largest) > 1 else 0
print(f'Connected components: {len(components)}. Path length uses the largest ({len(largest)}/{len(G)} nodes).')

print("=" * 60)
print("CO-PRESENCE CENTRALITY TABLE")
print("=" * 60)
pd.set_option('display.float_format', '{:.3f}'.format)
print(co_presence_df.to_string(index=False))

print("\n" + "=" * 60)
print("GLOBAL METRICS")
print("=" * 60)
print(f"Transitivity (clustering): {transitivity:.3f}")
print(f"Average Path Length (largest component): {avg_path:.3f}")

### Interpret
Compare the degree and strength leaders. Does having many partners mean having the most shared-scene links?


---
📊 **ANALYSIS QUESTION 1: Global Metrics**

What do your Transitivity and Average Path Length values reveal about your network's structure? Is it tightly clustered or sparse? Compact or spread out?

---
📊 **ANALYSIS QUESTION 2: Top Characters**

Who are the top 3 characters in each metric (Strength, Betweenness, Closeness, Eigenvector)? What does each ranking reveal about their narrative role?

---
📊 **ANALYSIS QUESTION 3: Metric Divergences**

Find 2+ characters who rank HIGH in one metric but LOW in another. What does this divergence reveal about their structural role?

## 1.2 Visualization 1: Community Detection

In [ ]:
# === Community Detection ===
groups = nx.community.louvain_communities(G, weight='Weight', seed=42)
partition = {node: cid for cid, members in enumerate(groups) for node in members}
num_communities = max(partition.values()) + 1
cmap = plt.get_cmap('Set2', num_communities)
node_colors = [cmap(partition[n]) for n in G.nodes()]

# === Visualization ===
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.8, seed=42)
edge_weights = [G[u][v]['Weight'] * 0.4 for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=700, node_color=node_colors, alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, edge_color='gray')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

plt.title("Visualization 1: Co-Presence Network with Communities", fontsize=14, fontweight='bold')
plt.figtext(0.5, 0.01, 'Color = detected community; line width = shared-scene count. Positions are a layout, not geography.', ha='center', fontsize=9)
plt.axis('off')
plt.tight_layout()
plt.show()

# Print communities
print("\nDetected Communities:")
communities = {}
for node, comm in partition.items():
    communities.setdefault(comm, []).append(node)
for cid, members in sorted(communities.items()):
    print(f"  Community {cid}: {', '.join(members)}")

### Interpret
Inspect each detected group. What unites its members? Treat a community as an algorithmic grouping to investigate, not a ready-made narrative category.


---
📊 **ANALYSIS QUESTION 4: Community Interpretation**

List each community and explain what unites its members. Which characters bridge between communities?

---
📊 **ANALYSIS QUESTION 5: Network Topology**

What is the overall SHAPE of your network? (Star/hub-spoke, barbell, core-periphery, small-world, linear chain, complete/dense?) Cite visual evidence and metrics.

## 1.3 Visualization 2: Filtered Network (Core Relationships)

In [ ]:
threshold = 2  # Change this, then rerun.
# Retain links with weights above the threshold.
G_filtered = nx.Graph()
for u, v, d in G.edges(data=True):
    if d['Weight'] > threshold:
        G_filtered.add_edge(u, v, **d)

# Visualization
plt.figure(figsize=(14, 10))
pos_f = nx.spring_layout(G_filtered, k=0.7, seed=42)
filtered_colors = [cmap(partition.get(n, 0)) for n in G_filtered.nodes()]
filtered_weights = [G_filtered[u][v]['Weight'] * 0.8 for u, v in G_filtered.edges()]

nx.draw_networkx_nodes(G_filtered, pos_f, node_size=700, node_color=filtered_colors, alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G_filtered, pos_f, width=filtered_weights, alpha=0.7, edge_color='darkred')
nx.draw_networkx_labels(G_filtered, pos_f, font_size=8, font_weight='bold')

plt.title(f"Visualization 2: Shared-scene count > {threshold}", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\nFiltered: {G_filtered.number_of_nodes()} nodes, {G_filtered.number_of_edges()} edges")
print(f"Removed {G.number_of_nodes() - G_filtered.number_of_nodes()} peripheral characters")

### Interpret
Compare before and after filtering. Which relationships disappear? A weak or infrequent connection can still matter to the story.


---
📊 **ANALYSIS QUESTION 6: Narrative Backbone**

Which characters/connections disappeared? What does this reveal about core vs. peripheral relationships? What are the thickest remaining edges?

## 1.4 Visualization 3: Betweenness Centrality (Node Size)

In [ ]:
# Node sizes from betweenness
max_b = max(betweenness.values()) if max(betweenness.values()) > 0 else 1
node_sizes = [300 + (betweenness[n] / max_b) * 2500 for n in G.nodes()]

plt.figure(figsize=(14, 10))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.4, edge_color='gray')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

plt.title("Visualization 3: Node Size = Betweenness Centrality", fontsize=14, fontweight='bold')
plt.figtext(0.5, 0.01, 'Larger node = higher betweenness; color = community, not importance.', ha='center', fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.show()

### Interpret
Identify a high-betweenness character and inspect the paths through it. Check the screenplay before interpreting this as a narrative bridge.


---
📊 **ANALYSIS QUESTION 7: Bridge Characters**

Which characters have the largest nodes? What communities do they connect? How does their position enable plot progression?

# 2. Dialogue
In-degree counts distinct speakers addressing a character; out-degree counts distinct addressees. **In-strength/out-strength** sum addressed speech turns. Weighted incoming eigenvector scores reward receiving connections from other central nodes; they do not measure a speaker's influence on listeners.


## 2.1 Dialogue Centrality Metrics

In [ ]:
in_degree = dict(G_dialogue.in_degree())
out_degree = dict(G_dialogue.out_degree())
in_strength = dict(G_dialogue.in_degree(weight='Weight'))
out_strength = dict(G_dialogue.out_degree(weight='Weight'))
try:
    eig_d = nx.eigenvector_centrality(G_dialogue, weight='Weight', max_iter=5000)
except nx.PowerIterationFailedConvergence:
    eig_d = {node: np.nan for node in G_dialogue}
    print('Eigenvector did not converge; shown as missing, not zero.')

dialogue_df = pd.DataFrame({
    'Character': list(G_dialogue),
    'In-Degree': [in_degree[n] for n in G_dialogue],
    'Out-Degree': [out_degree[n] for n in G_dialogue],
    'In-Strength': [in_strength[n] for n in G_dialogue],
    'Out-Strength': [out_strength[n] for n in G_dialogue],
    'Dialogue Eigenvector': [eig_d[n] for n in G_dialogue]
})
dialogue_df['In/Out Strength Ratio'] = dialogue_df['In-Strength'].div(
    dialogue_df['Out-Strength'].replace(0, np.nan))
dialogue_df = dialogue_df.sort_values('In-Strength', ascending=False)
display(dialogue_df.round(3))


### Interpret
Separate number of partners from number of turns. An imbalance can suggest a question about attention; it does not establish desire, authority, or motivation.


---
📊 **ANALYSIS QUESTION 8: Dialogue Patterns**

Identify: (a) The dialogue HUB (highest In-Strength), (b) The dialogue DRIVER (highest Out-Strength), (c) Any character with balanced In/Out, (d) High Eigenvector but low degree.

## 2.2 Visualization 4: Directed Dialogue Network

In [ ]:
plt.figure(figsize=(14, 10))
pos_d = nx.spring_layout(G_dialogue, k=0.7, seed=42)
d_weights = [G_dialogue[u][v]['Weight'] * 0.12 for u, v in G_dialogue.edges()]

nx.draw_networkx_nodes(G_dialogue, pos_d, node_size=600, node_color='skyblue', alpha=0.9, edgecolors='black')
nx.draw_networkx_edges(G_dialogue, pos_d, width=d_weights, alpha=0.6, edge_color='darkblue',
                       arrowstyle='->', arrowsize=15, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G_dialogue, pos_d, font_size=8, font_weight='bold')

plt.title("Visualization 4: Directed Dialogue Network (Arrow = Speaker → Addressee)", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

### Interpret
Follow one pair of opposite arrows. How does the direction change the relationship the graph represents?


---
📊 **ANALYSIS QUESTION 9: Arrow Patterns**

Find: (a) A character with many INCOMING arrows (has many speakers addressing them), (b) A character with many OUTGOING arrows (has many addressees), (c) An asymmetric/unreciprocated relationship.

## 2.3 Visualization 5: In-Degree vs Out-Degree Scatter

In [ ]:
plt.figure(figsize=(12, 10))

# Scatter
plt.scatter(dialogue_df['Out-Strength'], dialogue_df['In-Strength'],
            s=dialogue_df['Dialogue Eigenvector'].fillna(0)*1500 + 100, alpha=0.6, c='steelblue', edgecolors='black')

# Labels
for _, row in dialogue_df.iterrows():
    plt.annotate(row['Character'], (row['Out-Strength'], row['In-Strength']), fontsize=8, ha='left')

# Diagonal
max_val = max(dialogue_df['In-Strength'].max(), dialogue_df['Out-Strength'].max())
plt.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='Balanced (In = Out)')

plt.xlabel('Out-strength (addressed speech turns sent)', fontsize=12)
plt.ylabel('In-strength (addressed speech turns received)', fontsize=12)
plt.title('Visualization 5: Dialogue in-strength vs out-strength\n(Size = Eigenvector Centrality)', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

### Interpret
Above the diagonal means more addressed turns received than sent. What passage could explain that imbalance? The graph alone does not explain its emotional meaning.


---
📊 **ANALYSIS QUESTION 10: Scatter Plot Analysis**

Who is above the diagonal (spoken to > speaks)? Who is below (speaks > spoken to)? What does this reveal about power dynamics or narrative roles?

---
# Session 3: Comprehensive Comparison

In [ ]:
# Missing in a network means not represented in that edge list, not necessarily zero.
full_df = pd.merge(co_presence_df, dialogue_df, on='Character', how='outer')
display(full_df.sort_values('Strength', ascending=False).round(3))


### Interpret
Find a character whose degree, strength, or dialogue rankings differ. Use that discrepancy to choose a passage to investigate.


---
# Final Synthesis: Three Arguments

📊 **ARGUMENT 1: Network Topology**

Make a claim about your network's overall structure (e.g., "two-protagonist network," "hub-and-spoke," "fragmented communities"). Support with: (a) specific visualization, (b) quantitative metrics, (c) narrative interpretation.

---

📊 **ARGUMENT 2: Metric Divergence**

Identify a character whose metrics diverge (high in X, low in Y). Explain what this reveals about their narrative function using both visual and quantitative evidence.

---

📊 **ARGUMENT 3: Dialogue Asymmetry**

Find a relationship where dialogue is asymmetric (A speaks to B more than B speaks to A). What does this reveal about power, desire, or narrative tension?

### Interpret
Make one claim you can support with a table, a graph, and a specific scene. Distinguish what the edge list measures from what you infer about the story.
